# Part 4: Machine Learning Model for Bridge Vulnerability Prediction

This part of the workflow develops a machine learning model to predict bridge vulnerability using bridge-specific structural and condition-related variables. The main purpose of this step is to create a data-driven vulnerability model that complements the HAZUS fragility workflow while remaining focused on intrinsic bridge characteristics.

The model is designed deliberately without PGA as an input feature. This is because PGA represents earthquake hazard intensity rather than the inherent structural vulnerability of a bridge. Including PGA would make the model overly dependent on event severity and reduce its usefulness as a bridge-specific vulnerability predictor. Instead, the machine learning model uses variables that reflect the bridge itself.

The selected input features are:
- year built
- reconstructed year
- number of spans
- maximum span length
- skew
- condition rating
- HAZUS bridge class
- SVI

These variables were chosen because they are structurally meaningful and align with engineering intuition. Together, they capture age, maintenance history, geometry, condition, classification, and continuous vulnerability.

The target variable used here is Expected Damage Ratio (EDR), which was computed earlier through the HAZUS fragility framework. In this way, the machine learning model learns to approximate bridge damage susceptibility from bridge-specific variables alone, without relying directly on hazard intensity.

This step is important because it helps examine whether structural and condition-related factors can explain damage patterns in a more flexible and interpretable way. It also creates the foundation for a future predictive dashboard or web-based tool.

## How to use this notebook

This notebook trains regression models to predict Expected Damage Ratio using the processed bridge and hazard features. It is best understood as a comparison and extension layer on top of the engineering workflow, not a replacement for the HAZUS logic.

**Input**
- `data/processed/bridges_with_svi.csv`

**Output**
- `data/processed/bridge_ml_predictions.csv`

**Why this step matters**
It tests whether bridge vulnerability patterns learned from the HAZUS-based workflow can be reproduced efficiently with standard predictive models.

**Models tested**
- Linear Regression
- Random Forest
- Gradient Boosting


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor

from project_paths import build_paths, require_paths

PATHS = build_paths()
INPUT_CSV = PATHS["SVI_CSV"]
OUT_ML_PRED_CSV = PATHS["ML_PREDICTIONS_CSV"]

require_paths(PATHS, ["SVI_CSV"])

df = pd.read_csv(INPUT_CSV, low_memory=False)

feature_cols = [
    "year",
    "yr_recon",
    "spans",
    "max_span",
    "skew",
    "cond",
    "SVI",
    "HWB_CLASS",
]
target_col = "EDR"

model_df = df[feature_cols + [target_col]].copy()
model_df = model_df.dropna(subset=[target_col]).copy()

X = model_df[feature_cols]
y = model_df[target_col]

numeric_features = [
    "year",
    "yr_recon",
    "spans",
    "max_span",
    "skew",
    "cond",
    "SVI",
]
categorical_features = ["HWB_CLASS"]

numeric_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

categorical_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore")),
])

preprocessor = ColumnTransformer([
    ("num", numeric_transformer, numeric_features),
    ("cat", categorical_transformer, categorical_features),
])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

models = {
    "Linear Regression": LinearRegression(),
    "Random Forest": RandomForestRegressor(
        n_estimators=200,
        max_depth=12,
        min_samples_split=10,
        min_samples_leaf=4,
        random_state=42,
        n_jobs=-1,
    ),
    "Gradient Boosting": GradientBoostingRegressor(
        n_estimators=200,
        learning_rate=0.05,
        max_depth=3,
        random_state=42,
    ),
}

results = []
predictions_dict = {}

for name, model in models.items():
    pipe = Pipeline([
        ("preprocessor", preprocessor),
        ("model", model),
    ])
    pipe.fit(X_train, y_train)
    y_pred = pipe.predict(X_test)

    mae = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    r2 = r2_score(y_test, y_pred)

    results.append({
        "Model": name,
        "MAE": mae,
        "RMSE": rmse,
        "R2": r2,
    })
    predictions_dict[name] = {
        "pipeline": pipe,
        "y_pred": y_pred,
    }

results_df = pd.DataFrame(results).sort_values("RMSE")
print(results_df)

best_model_name = results_df.iloc[0]["Model"]
best_pipe = predictions_dict[best_model_name]["pipeline"]
best_pred = predictions_dict[best_model_name]["y_pred"]

pred_df = X_test.copy()
pred_df["Actual_EDR"] = y_test.values
pred_df["Predicted_EDR"] = best_pred
pred_df["Best_Model"] = best_model_name
pred_df.to_csv(OUT_ML_PRED_CSV, index=False)

print("\nBest model:", best_model_name)
print("\nSaved:", OUT_ML_PRED_CSV)
print("\nPrediction sample:")
print(pred_df.head())


## Inference

The machine learning results show that the Random Forest model performed best among the tested approaches, with lower MAE and RMSE and a higher R² than both Gradient Boosting and Linear Regression. This suggests that the relationship between bridge characteristics and Expected Damage Ratio is not purely linear and is better captured by a flexible nonlinear model. That result is reasonable in the context of bridge vulnerability, where damage behavior depends on interactions among structural age, condition, geometry, and bridge type.

The selected input variables also make engineering sense. Year built and reconstructed year reflect age and retrofit history, spans and maximum span length capture bridge geometry, skew reflects structural irregularity, condition rating reflects present structural health, HAZUS class preserves engineering-based bridge categorization, and SVI provides a continuous bridge-specific vulnerability measure. Together, these features create a model that focuses on intrinsic bridge vulnerability rather than simply reproducing hazard intensity.

The fact that the best-performing model was Random Forest also indicates that vulnerability is influenced by combinations of factors rather than by one variable in isolation. Overall, this step successfully extends the project from an engineering fragility workflow into a data-driven predictive framework while still keeping the feature selection interpretable and structurally meaningful.

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 6))
plt.scatter(y_test, best_pred, s=12, alpha=0.5)
plt.xlabel("Actual EDR")
plt.ylabel("Predicted EDR")
plt.title(f"Actual vs Predicted EDR ({best_model_name})")
plt.grid(True)
plt.show()
